# 卷积神经网络GoogLeNet
之前的网络中，卷积核从 $11 \times 11$ 到 $3 \times 3$ 大小的卷积核都有使用，但是哪一种大小的卷积核最合适呢？GoogLeNet使用了特殊的结构Inception块，将不同大小的卷积核并行运算，最后按通道方向合并，以此来解决什么样大小的卷积核最合适的问题。

## Inception块
在GoogLeNet中，基本的卷积块被称为Inception块（Inception block）。

![图1：Inception块的架构。](https://s2.loli.net/2025/07/19/7bHN8XdL5MVeq4O.png)

如图1所示，Inception块有四条并行的计算路径，分别是 $1 \times 1$、$3 \times 3$、$5 \times 5$的卷积层和一个 $3 \times 3$ 的最大汇聚层，图中额外的 $1 \times 1$ 的卷积层是用来减少通道数的，称为Reduce层。这四条路径最后都输出的高和宽一致，最后合并到一起。

实现一个Inception块需要接收`in_channels, ch1x1, ch3x3red, ch3x3, ch5x5red, ch5x5, pool_proj`这几个参数，分别代表输入通道数、$1 \times 1$ 卷积层的输出通道数、$3 \times 3$ 卷积层前置Reduce层的输出通道数、$3 \times 3$ 卷积层的输出通道数、$5 \times 5$ 卷积层前置Reduce层的输出通道数、$5 \times 5$ 卷积层的输出通道数、池化层的输出通道数。

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
import numpy as np
import os
import time
from training_visualizer import TrainingPlotter

In [ ]:
class BasicConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super(BasicConv2d, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, **kwargs)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        return x

class Inception(nn.Module):
    def __init__(self, in_channels, ch1x1, ch3x3red, ch3x3, ch5x5red, ch5x5, pool_proj):
        super(Inception, self).__init__()

        self.branch1 = BasicConv2d(in_channels, ch1x1, kernel_size=1)

        self.branch2 = nn.Sequential(
            BasicConv2d(in_channels, ch3x3red, kernel_size=1),
            BasicConv2d(ch3x3red, ch3x3, kernel_size=3, padding=1)
        )

        self.branch3 = nn.Sequential(
            BasicConv2d(in_channels, ch5x5red, kernel_size=1),
            BasicConv2d(ch5x5red, ch5x5, kernel_size=5, padding=2)
        )

        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            BasicConv2d(in_channels, pool_proj, kernel_size=1)
        )

    def forward(self, x):
        branch1 = self.branch1(x)
        branch2 = self.branch2(x)
        branch3 = self.branch3(x)
        branch4 = self.branch4(x)

        outputs = [branch1, branch2, branch3, branch4]

        return torch.cat(outputs, dim=1)

在GoogLeNet结构中，还有两个辅助分类器在inception(4a)和inception(4a)处。

![图2：辅助分类器的结构。](https://s2.loli.net/2025/07/10/YnB89sHI3FOMGKy.png)

In [ ]:
class InceptionAux(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(InceptionAux, self).__init__()

        self.averagePool = nn.AvgPool2d(kernel_size=5, stride=3)
        self.conv = BasicConv2d(in_channels, 128, kernel_size=1)

        self.fc1 = nn.Linear(2048, 1024)
        self.fc2 = nn.Linear(1024, num_classes)
    
    def forward(self, x):
        # 辅助分类器1：N x 512 x 14 x 14，辅助分类器2：N x 528 x 14 x 14
        x = self.averagePool(x)
        # 辅助分类器1：N x 512 x 4 x 4，辅助分类器2：N x 528 x 4 x 4
        x = self.conv(x)
        # N x 128 x 4 x 4
        x = torch.flatten(x, 1)
        x = F.dropout(x, 0.5, training=self.training)
        # N x 2048
        x = F.relu(self.fc1(x), inplace=True)
        x = F.dropout(x, 0.5, training=self.training)
        # N x 1024
        x = self.fc2(x)
        # N x num_classes
        return x


## GoogLeNet结构
GoogLeNet一共使用了9个Inception块，最后用了一个平均汇聚层来避免全连接层的使用（类似NiN中使用 $1 \times 1$ 的卷积层来代替全连接层）。

![图3：GoogLeNet架构](https://s2.loli.net/2025/07/10/vQMHgtCiWIJO7qP.jpg)

GoogLeNet结构比较复杂，若将各层参数列成一张表格。则

![图4：GoogLeNet各层参数](https://s2.loli.net/2025/07/10/9LwBROudhC3K1MZ.jpg)

In [ ]:
class GoogLeNet(nn.Module):
    def __init__(self, num_classes=1000, aux_logits=True, init_weight=False):
        super(GoogLeNet, self).__init__()
        self.aux_logits = aux_logits

        # GoogLeNet包含五个模块，每个模块后面紧跟一个池化层
        # 第一个模块包含1个卷积层 
        self.conv1 = BasicConv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.maxpool1 = nn.MaxPool2d(3, stride=2, ceil_mode=True)   # 高宽减少至1/4
        # 第二个模块包含2个卷积层
        self.conv2 = BasicConv2d(64, 64, kernel_size=1)
        self.conv3 = BasicConv2d(64, 192, kernel_size=3, padding=1) 
        self.maxpool2 = nn.MaxPool2d(3, stride=2, ceil_mode=True)   # 通道数64->192 高宽减半
        # 第三个模块包含2个Inception块
        self.inception3a = Inception(192, 64, 96, 128, 16, 32, 32)
        self.inception3b = Inception(256, 128, 128, 192, 32, 96, 64)
        self.maxpool3 = nn.MaxPool2d(3, stride=2, ceil_mode=True)   # 通道数 192->256->480 高宽减半
        # 第四个模块包含5个Inception块
        self.inception4a = Inception(480, 192, 96, 208, 16, 48, 64)
        self.inception4b = Inception(512, 160, 112, 224, 24, 64, 64)
        self.inception4c = Inception(512, 128, 128, 256, 24, 64, 64)
        self.inception4d = Inception(512, 112, 144, 288, 32, 64, 64)
        self.inception4e = Inception(528, 256, 160, 320, 32, 128, 128)
        self.maxpool4 = nn.MaxPool2d(3, stride=2, ceil_mode=True)   # 通道数480->832 高宽减半
        # 第五个模块包含2个Inception块
        self.inception5a = Inception(832, 256, 160, 320, 32, 128, 128)
        self.inception5b = Inception(832, 384, 192, 384, 48, 128, 128)  # 通道数832->1024
        # 两个辅助输出层
        if self.aux_logits:
            self.aux1 = InceptionAux(512, num_classes)
            self.aux2 = InceptionAux(528, num_classes)
        # 全局池化层
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1)) # 不管最后形状如何，都汇聚成1x1的长条，替代全连接层
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(1024, num_classes)
        if init_weight:
            self._initialize_weights()

    def forward(self, x):
        aux1, aux2 = 0, 0
        # N x 3 x 224 x 224
        x = self.conv1(x)
        # N x 64 x 112 x 112
        x = self.maxpool1(x)
        # N x 64 x 56 x56
        x = self.conv2(x)
        # N x 64 x 56 x 56
        x = self.conv3(x)
        # N x 192 x 56 x 56
        x = self.maxpool2(x)

        # N x 192 x 28 x 28
        x = self.inception3a(x)
        # N x 256 x 28 x 28
        x = self.inception3b(x)
        # N x 480 x 28 x 28
        x = self.maxpool3(x)

        # N x 480 x 14 x 14
        x = self.inception4a(x)
        # N x 512 x 14 x 14
        if self.training and self.aux_logits:    # 辅助层只需在训练时启用
            aux1 = self.aux1(x)
        x = self.inception4b(x)
        # N x 512 x 14 x 14
        x = self.inception4c(x)
        # N x 512 x 14 x 14
        x = self.inception4d(x)
        # N x 528 x 14 x 14
        if self.training and self.aux_logits:
            aux2 = self.aux2(x)
        x = self.inception4e(x)
        # N x 832 x 14 x 14
        x = self.maxpool4(x)

        # N x 832 x 7 x 7
        x = self.inception5a(x)
        # N x 832 x 7 x 7
        x = self.inception5b(x)
        # N x 1024 x 7 x 7
        x = self.avgpool(x)
        # N x 1024 x 1 x 1
        x = torch.flatten(x, 1)
        # N x 1024
        x = self.dropout(x)
        x = self.fc(x)
        # N x 1000 (num_classes)
        if self.training and self.aux_logits:
            return x, aux2, aux1
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

## 加载数据集
这里使用 Oxford-IIIT Pet Dataset 数据集来训练，它有37个宠物品种的类，每个类大概200张图片。

In [ ]:
BATCH_SIZE = 64
# 数据集存放目录
DATASET_DIR = r'data/food-101'

if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)

# 1. 定义数据预处理和增强
train_transform = transforms.Compose([
    transforms.Resize(256),                     # 调整大小
    transforms.RandomCrop(224),                 # 随机裁剪
    transforms.RandomHorizontalFlip(),          # 随机水平翻转
    transforms.RandomRotation(10),              # 随机旋转
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # 颜色抖动
    transforms.ToTensor(),                      # 转换为张量
    transforms.Normalize(                       # 标准化
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])
# 2. 设置数据集路径
test_transform = transforms.Compose([
    transforms.Resize(256),                     
    transforms.CenterCrop(224),                 # 中心裁剪
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.Food101(
    root=DATASET_DIR, 
    split='train', 
    download=True, 
    transform=train_transform
)

val_dataset = datasets.Food101(
    root=DATASET_DIR, 
    split='test', 
    download=True, 
    transform=test_transform
)
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

## 开始训练


In [ ]:
# 配置训练参数

EPOCHS = 50
NUM_CLASSES = 101  # Food101 有 101 个类别
IMAGE_SIZE = 224   #  标准输入尺寸


# 模型参数存放目录
MODEL_DIR = r'/kaggle/working/'
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

In [ ]:
# 设置训练设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 初始化模型
model = GoogLeNet(num_classes=NUM_CLASSES, aux_logits=True, init_weight=True)
model = model.to(device)

# 定义损失函数和优化器
loss_fn = nn.CrossEntropyLoss().to(device)

# 设置 SGD with Momentum 优化器（论文核心参数）
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.045,           # 初始学习率 (论文未明确但后续 Inception 论文确认为 ~0.045)
    momentum=0.9,       # 动量因子 (论文明确指定)
    weight_decay=2e-4   # 权重衰减 (论文明确指定为 0.0002)
)

# 3. 定义学习率调度器（论文策略：每 8 个 epoch 下降 4%）
scheduler = StepLR(
    optimizer,
    step_size=8,        # 每 8 个 epoch 更新一次学习率
    gamma=0.96          # 学习率衰减因子 (1 - 0.04 = 0.96)
)

# epoch总数
num_epochs = EPOCHS

# 总数据集大小
train_data_size = len(train_dataset)
test_data_size = len(val_dataset)
# 每个epoch内部循环次数
train_iter_size = len(train_loader)
test_iter_size = len(val_loader)
print(f'Training dataset size: {train_data_size}, Validation dataset size: {test_data_size}')

# 训练和验证循环
best_acc = 0
plotter = TrainingPlotter()
for epoch in range(num_epochs):
    start_time = time.time()
    # 训练阶段
    model.train()
    total_train_accuracy = 0
    total_train_loss = 0
    for train_batch, label in train_loader:
        train_batch, label = train_batch.to(device), label.to(device)
        output = model(train_batch)
        loss = loss_fn(output, label)
        # 优化器模型
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # 累计损失率
        total_train_loss += loss
        # 累计准确度
        accuracy = (output.argmax(1) == label).sum()
        total_train_accuracy += accuracy.item()
    
    # 测试阶段
    model.eval()
    total_test_accuracy = 0
    with torch.no_grad():
        for test_batch, label in val_loader:
            test_batch, label = test_batch.to(device), label.to(device)
            output = model(test_batch)
            accuracy = (output.argmax(1) == label).sum()
            total_test_accuracy += accuracy.item()


    train_loss = total_train_loss/train_iter_size
    train_acc = total_train_accuracy/train_data_size
    test_acc = total_test_accuracy/test_data_size
    # 绘制图表
    plotter.update(
        epoch,
        train_loss,
        train_acc,
        test_acc
    )
   
    # 每个epoch结束后更新学习率
    scheduler.step()

    # 一个周期结束
    end_time = time.time()
    print(f"epoch：{epoch}，训练集损失：{train_loss}，训练集准确度：{train_acc}，测试集准确度：{test_acc}")
    print(f"训练耗时：{(end_time - start_time):2f}")
    if best_acc < test_acc:
        best_acc = test_acc
        pth_save_path = os.path.join(MODEL_DIR, f"GoogLeNet_best.pth")
        torch.save(model.state_dict(), pth_save_path)

plotter.final_report()

batch_size=128 lr=0.003 的训练结果

![图6：训练结果](https://s2.loli.net/2025/07/10/WXfkdeM2zHt4GQw.png)

如果设置 batch_size=32 lr=3e-4 效果会更好，图片没保存，kaggle免费时长不够了，不再试了。